# ==========================================================
# Breast Cancer Survival Prediction using Apache Spark
# Notebook 02: Data Preprocessing
# ==========================================================

"""
Objective
---------
1. Clean the raw SEER breast cancer dataset.
2. Handle SEER special codes.
3. Handle missing values.
4. Remove invalid records.
5. Standardize variables.
6. Prepare a clean dataset for feature engineering.

Note
----
No feature engineering.
No model training.
No model evaluation.
"""

In [ ]:
# 1. Import Libraries

import os
import sys

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

# Add project root directory

PROJECT_ROOT = os.path.abspath("..")

if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

# Import custom functions

from src.data.loader import (
    create_spark_session,
    load_csv
)

In [2]:
# 2. Create Spark Session

spark = create_spark_session(
    "SEER Breast Cancer Data Preprocessing"
)

In [3]:
# 3. Load Raw CSV Dataset

df = load_csv(
    spark,
    "../data/raw/breast_cancer_seer-2004-2015.csv"
)

# Store original dataset information before preprocessing

original_row_count = df.count()
original_column_count = len(df.columns)

print("=" * 60)
print("ORIGINAL DATASET INFORMATION")
print("=" * 60)

print(f"Original Rows    : {original_row_count:,}")
print(f"Original Columns : {original_column_count}")

ORIGINAL DATASET INFORMATION
Original Rows    : 457,351
Original Columns : 29


In [4]:
# 4. Review Dataset Structure

print("=" * 60)
print("RAW DATASET OVERVIEW")
print("=" * 60)

print(f"Number of rows    : {df.count():,}")
print(f"Number of columns : {len(df.columns)}")

print()

df.printSchema()

print()

df.show(10, truncate=False)

RAW DATASET OVERVIEW
Number of rows    : 457,351
Number of columns : 29

root
 |-- Age recode with <1 year olds and 90+: string (nullable = true)
 |-- Sex: string (nullable = true)
 |-- Race recode (W, B, AI, API): string (nullable = true)
 |-- Marital status at diagnosis: string (nullable = true)
 |-- CS tumor size (2004-2015): integer (nullable = true)
 |-- Survival months: string (nullable = true)
 |-- Vital status recode (study cutoff used): string (nullable = true)
 |-- Grade Recode (thru 2017): string (nullable = true)
 |-- PR Status Recode Breast Cancer (1990+): string (nullable = true)
 |-- ER Status Recode Breast Cancer (1990+): string (nullable = true)
 |-- Breast - Adjusted AJCC 6th T (1988-2015): string (nullable = true)
 |-- Breast - Adjusted AJCC 6th N (1988-2015): string (nullable = true)
 |-- Regional nodes examined (1988+): integer (nullable = true)
 |-- Regional nodes positive (1988+): integer (nullable = true)
 |-- Sequence number: string (nullable = true)
 |-- Patie

In [5]:
# ==========================================================
# 5. Rename Columns
# ==========================================================


column_mapping = {

"Age recode with <1 year olds and 90+": "Age",

"Sex": "Sex",

"Race recode (W, B, AI, API)": "Race",

"Marital status at diagnosis":
"Marital_Status",

"CS tumor size (2004-2015)":
"Tumor_Size",

"Survival months":
"Survival_Months",

"Vital status recode (study cutoff used)":
"Vital_Status",

"Grade Recode (thru 2017)":
"Grade",

"PR Status Recode Breast Cancer (1990+)":
"PR_Status",

"ER Status Recode Breast Cancer (1990+)":
"ER_Status",

"Breast - Adjusted AJCC 6th T (1988-2015)":
"AJCC_T",

"Breast - Adjusted AJCC 6th N (1988-2015)":
"AJCC_N",

"Breast - Adjusted AJCC 6th M (1988-2015)":
"AJCC_M",

"Breast - Adjusted AJCC 6th Stage (1988-2015)":
"AJCC_Stage",

"Regional nodes examined (1988+)":
"Regional_Nodes_Examined",

"Regional nodes positive (1988+)":
"Regional_Nodes_Positive",

"Sequence number":
"Sequence_Number",

"Patient ID":
"Patient_ID",

"Primary Site":
"Primary_Site",

"Histologic Type ICD-O-3":
"Histologic_Type",

"Behavior recode for analysis":
"Behavior",

"Laterality":
"Laterality",

"Diagnostic Confirmation":
"Diagnostic_Confirmation",

"Lymph-vascular Invasion (2004+ varying by schema)":
"Lymph_Vascular_Invasion",

"RX Summ--Surg Prim Site (1998-2022)":
"Surgery_Primary_Site",

"RX Summ--Surg Oth Reg/Dis (2003+)":
"Surgery_Other_Regional",

"RX Summ--Surg/Rad Seq":
"Surgery_Radiation_Sequence",

"Radiation recode":
"Radiation",

"Chemotherapy recode (yes, no/unk)":
"Chemotherapy"

}


for old,new in column_mapping.items():

    if old in df.columns:

        df = df.withColumnRenamed(
            old,
            new
        )


print("Column renaming completed.")

Column renaming completed.


In [6]:
# ==========================================================
# 6. Verify Column Renaming
# ==========================================================

print(
    df.columns
)

print(
    "Number of columns:",
    len(df.columns)
)

['Age', 'Sex', 'Race', 'Marital_Status', 'Tumor_Size', 'Survival_Months', 'Vital_Status', 'Grade', 'PR_Status', 'ER_Status', 'AJCC_T', 'AJCC_N', 'Regional_Nodes_Examined', 'Regional_Nodes_Positive', 'Sequence_Number', 'Patient_ID', 'Primary_Site', 'Histologic_Type', 'Behavior', 'Laterality', 'Diagnostic_Confirmation', 'AJCC_M', 'Lymph_Vascular_Invasion', 'Surgery_Primary_Site', 'Surgery_Other_Regional', 'Surgery_Radiation_Sequence', 'Radiation', 'Chemotherapy', 'AJCC_Stage']
Number of columns: 29


In [7]:
# ==========================================================
# 7. Check SEER Coding
# ==========================================================


check_columns = [
    "Tumor_Size",
    "Survival_Months",
    "Regional_Nodes_Examined",
    "Regional_Nodes_Positive"
]


for c in check_columns:

    print("\n")
    print("="*50)
    print(c)
    print("="*50)


    df.groupBy(c)\
      .count()\
      .orderBy(
          F.desc("count")
      )\
      .show(15,False)



Tumor_Size
+----------+-----+
|Tumor_Size|count|
+----------+-----+
|15        |27858|
|999       |27129|
|25        |19416|
|10        |19217|
|20        |18944|
|12        |18893|
|30        |15857|
|8         |14055|
|18        |14035|
|11        |13546|
|9         |12793|
|13        |12424|
|7         |11585|
|14        |11248|
|17        |11191|
+----------+-----+
only showing top 15 rows



Survival_Months
+---------------+-----+
|Survival_Months|count|
+---------------+-----+
|0000           |4160 |
|0098           |3800 |
|0102           |3718 |
|0105           |3710 |
|0096           |3676 |
|0100           |3613 |
|0097           |3581 |
|0107           |3578 |
|0099           |3512 |
|0101           |3509 |
|0104           |3499 |
|0108           |3452 |
|0103           |3433 |
|0116           |3415 |
|0110           |3344 |
+---------------+-----+
only showing top 15 rows



Regional_Nodes_Examined
+-----------------------+-----+
|Regional_Nodes_Examined|count|
+---------

In [ ]:
# ==========================================================
# 8. Handle SEER Special Codes
# ==========================================================

print("=" * 60)
print("HANDLING SEER SPECIAL CODES")
print("=" * 60)

# ==========================================================
# Numeric SEER special codes
# ==========================================================

numeric_special_codes = {

    "Tumor_Size": [
        990,
        991,
        992,
        993,
        994,
        995,
        996,
        997,
        998,
        999
    ],

    "Regional_Nodes_Examined": [
        90,
        95,
        96,
        97,
        98,
        99
    ],

    "Regional_Nodes_Positive": [
        90,
        95,
        96,
        97,
        98,
        99
    ]
}

for column, codes in numeric_special_codes.items():
    if column in df.columns:
        count_before = (
            df
            .filter(
                F.col(column).isin(codes)
            )
            .count()
        )

        df = df.withColumn(
            column,
            F.when(
                F.col(column).isin(codes),
                None
            )
            .otherwise(
                F.col(column)
            )
        )

        print(f"{column}: {count_before:,} special values converted")

# ==========================================================
# String SEER special codes
# ==========================================================

string_special_codes = [

    "Unknown",
    "Unknown reason",
    "Blank(s)",
    "NA",
    "Not Applicable",
    "Unspecified"
]

for column, dtype in df.dtypes:
    if dtype == "string":
        count_before = (
            df
            .filter(
                F.col(column).isin(string_special_codes)
            )
            .count()
        )

        if count_before > 0:

            df = df.withColumn(
                column,
                F.when(
                    F.col(column).isin(
                        string_special_codes
                    ),
                    None
                )
                .otherwise(
                    F.col(column)
                )
            )

            print(f"{column}: {count_before:,} special values converted")

print("\nSEER special codes converted to NULL.")

HANDLING SEER SPECIAL CODES
Tumor_Size: 37,004 special values converted
Regional_Nodes_Examined: 19,874 special values converted
Regional_Nodes_Positive: 73,774 special values converted
Race: 2,499 special values converted
Marital_Status: 22,144 special values converted
Survival_Months: 2,753 special values converted
Grade: 49,354 special values converted
AJCC_T: 485 special values converted
AJCC_N: 485 special values converted
Diagnostic_Confirmation: 4,412 special values converted
AJCC_M: 485 special values converted
Lymph_Vascular_Invasion: 457,351 special values converted
AJCC_Stage: 485 special values converted

SEER special codes converted to NULL.


In [9]:
# ==========================================================
# 9. Missing Check After Special Code Handling
# ==========================================================

print("=" * 60)
print("NULL CHECK AFTER SPECIAL CODE HANDLING")
print("=" * 60)

missing_counts_after_special = {}
for column in df.columns:
    count = (
        df
        .filter(
            F.col(column).isNull()
        )
        .count()
    )
    missing_counts_after_special[column] = count
    if count > 0:

        print(
            f"{column:40} {count:,}"
        )

NULL CHECK AFTER SPECIAL CODE HANDLING
Race                                     2,499
Marital_Status                           22,144
Tumor_Size                               37,004
Survival_Months                          2,753
Grade                                    49,354
AJCC_T                                   485
AJCC_N                                   485
Regional_Nodes_Examined                  19,874
Regional_Nodes_Positive                  73,774
Diagnostic_Confirmation                  4,412
AJCC_M                                   485
Lymph_Vascular_Invasion                  457,351
AJCC_Stage                               485


In [10]:
# ==========================================================
# 10. Missing Value Percentage After Cleaning
# ==========================================================

print("=" * 60)
print("MISSING VALUE PERCENTAGE AFTER CLEANING")
print("=" * 60)

total_rows = df.count()
for column in df.columns:

    missing_count = (
        df
        .filter(
            F.col(column).isNull()
        )
        .count()
    )

    percentage = (
        missing_count / total_rows * 100
    )

    if missing_count > 0:

        print(
            f"{column:40} "
            f"{missing_count:10,} "
            f"({percentage:6.2f}%)"
        )

MISSING VALUE PERCENTAGE AFTER CLEANING
Race                                          2,499 (  0.55%)
Marital_Status                               22,144 (  4.84%)
Tumor_Size                                   37,004 (  8.09%)
Survival_Months                               2,753 (  0.60%)
Grade                                        49,354 ( 10.79%)
AJCC_T                                          485 (  0.11%)
AJCC_N                                          485 (  0.11%)
Regional_Nodes_Examined                      19,874 (  4.35%)
Regional_Nodes_Positive                      73,774 ( 16.13%)
Diagnostic_Confirmation                       4,412 (  0.96%)
AJCC_M                                          485 (  0.11%)
Lymph_Vascular_Invasion                     457,351 (100.00%)
AJCC_Stage                                      485 (  0.11%)


In [11]:
# ==========================================================
# 11. Handle Missing Values
# ==========================================================

print("=" * 60)
print("HANDLING MISSING VALUES")
print("=" * 60)

categorical_columns = [
    "Race",
    "Marital_Status",
    "Grade",
    "Diagnostic_Confirmation"
]
for column in categorical_columns:
    if column in df.columns:
        df = df.fillna({column: "Unknown"})
        
print("Selected categorical missing values filled.")

HANDLING MISSING VALUES
Selected categorical missing values filled.


In [12]:
# ==========================================================
# 12. Validate Missing Values After Treatment
# ==========================================================

print("=" * 60)
print("MISSING VALUE VALIDATION")
print("=" * 60)
remaining_missing = False

for column in df.columns:
    count = (
        df
        .filter(
            F.col(column).isNull()
        )
        .count()
    )

    if count > 0:
        remaining_missing = True
        print(
            f"{column:40} {count:,}"
        )

if not remaining_missing:
    print(
        "No missing values remain."
    )

MISSING VALUE VALIDATION
Tumor_Size                               37,004
Survival_Months                          2,753
AJCC_T                                   485
AJCC_N                                   485
Regional_Nodes_Examined                  19,874
Regional_Nodes_Positive                  73,774
AJCC_M                                   485
Lymph_Vascular_Invasion                  457,351
AJCC_Stage                               485


In [13]:
# ==========================================================
# 13. Remove Invalid Records
# ==========================================================

print("=" * 60)
print("REMOVING INVALID RECORDS")
print("=" * 60)

before_count = df.count()
df = df.filter(
    (F.col("Tumor_Size").isNull()) |
    (
        (F.col("Tumor_Size") > 0) &
        (F.col("Tumor_Size") < 989)
    )
)

df = df.filter(
    (F.col("Regional_Nodes_Examined").isNull()) |
    (F.col("Regional_Nodes_Examined") >= 0)
)

df = df.filter(
    (F.col("Regional_Nodes_Positive").isNull()) |
    (F.col("Regional_Nodes_Positive") >= 0)
)
after_count = df.count()

print(f"Removed records: {before_count-after_count:,}")

REMOVING INVALID RECORDS
Removed records: 1,264


In [14]:
# ==========================================================
# 14. Handle Outliers
# ==========================================================

print("=" * 60)
print("OUTLIER CHECK")
print("=" * 60)

numerical_columns = [
    "Tumor_Size",
    "Regional_Nodes_Examined",
    "Regional_Nodes_Positive"
]

for column in numerical_columns:
    if column in df.columns:
        print("\n")
        print(column)
        df.select(column)\
          .summary()\
          .show()

OUTLIER CHECK


Tumor_Size
+-------+------------------+
|summary|        Tumor_Size|
+-------+------------------+
|  count|            419083|
|   mean| 24.07345084386625|
| stddev|24.369512695507026|
|    min|                 1|
|    25%|                11|
|    50%|                18|
|    75%|                30|
|    max|               988|
+-------+------------------+



Regional_Nodes_Examined
+-------+-----------------------+
|summary|Regional_Nodes_Examined|
+-------+-----------------------+
|  count|                 436394|
|   mean|      5.996496285466803|
| stddev|     6.9772890748489385|
|    min|                      0|
|    25%|                      1|
|    50%|                      3|
|    75%|                      9|
|    max|                     87|
+-------+-----------------------+



Regional_Nodes_Positive
+-------+-----------------------+
|summary|Regional_Nodes_Positive|
+-------+-----------------------+
|  count|                 382962|
|   mean|     1.28331531587

In [15]:
# ==========================================================
# 15. Convert Data Types
# ==========================================================

print("=" * 60)
print("CONVERTING DATA TYPES")
print("=" * 60)

# ----------------------------------------------------------
# Convert Survival Months
# ----------------------------------------------------------

if "Survival_Months" in df.columns:
    df = df.withColumn(
        "Survival_Months",
        F.regexp_replace(
            F.col("Survival_Months"),
            "^0+",
            ""
        )
    )

    df = df.withColumn(
        "Survival_Months",
        F.when(
            F.col("Survival_Months") == "",
            None
        )
        .otherwise(
            F.col("Survival_Months")
        )
    )

    df = df.withColumn(
        "Survival_Months",
        F.col("Survival_Months").cast("int")
    )

# ----------------------------------------------------------
# Convert Numerical Variables
# ----------------------------------------------------------

numeric_columns = [
    "Tumor_Size",
    "Regional_Nodes_Examined",
    "Regional_Nodes_Positive"
]

for column in numeric_columns:
    if column in df.columns:
        df = df.withColumn(
            column,
            F.col(column).cast("int")
        )

print("Data type conversion completed.")

CONVERTING DATA TYPES
Data type conversion completed.


In [16]:
# ==========================================================
# Validate Data Types
# ==========================================================

print("=" * 60)
print("UPDATED SCHEMA")
print("=" * 60)

df.printSchema()

UPDATED SCHEMA
root
 |-- Age: string (nullable = true)
 |-- Sex: string (nullable = true)
 |-- Race: string (nullable = false)
 |-- Marital_Status: string (nullable = false)
 |-- Tumor_Size: integer (nullable = true)
 |-- Survival_Months: integer (nullable = true)
 |-- Vital_Status: string (nullable = true)
 |-- Grade: string (nullable = false)
 |-- PR_Status: string (nullable = true)
 |-- ER_Status: string (nullable = true)
 |-- AJCC_T: string (nullable = true)
 |-- AJCC_N: string (nullable = true)
 |-- Regional_Nodes_Examined: integer (nullable = true)
 |-- Regional_Nodes_Positive: integer (nullable = true)
 |-- Sequence_Number: string (nullable = true)
 |-- Patient_ID: integer (nullable = true)
 |-- Primary_Site: integer (nullable = true)
 |-- Histologic_Type: integer (nullable = true)
 |-- Behavior: string (nullable = true)
 |-- Laterality: string (nullable = true)
 |-- Diagnostic_Confirmation: string (nullable = false)
 |-- AJCC_M: string (nullable = true)
 |-- Lymph_Vascular_Inva

In [17]:
df.select(
    "Survival_Months"
).summary().show()

+-------+------------------+
|summary|   Survival_Months|
+-------+------------------+
|  count|            449219|
|   mean|123.41389389139819|
| stddev| 61.97242829847599|
|    min|                 1|
|    25%|                84|
|    50%|               126|
|    75%|               170|
|    max|               239|
+-------+------------------+



In [18]:
df.select(
    "AJCC_T",
    "AJCC_N",
    "AJCC_M",
    "AJCC_Stage"
).distinct().show(20, False)

+-----------+-----------+------+----------+
|AJCC_T     |AJCC_N     |AJCC_M|AJCC_Stage|
+-----------+-----------+------+----------+
|T4a        |N2         |M0    |IIIB      |
|T4c        |NX Adjusted|M0    |IIINOS    |
|Any T, Mets|NX Adjusted|M1    |IV        |
|T4d        |N3         |M0    |IIIC      |
|T1mic      |N1         |M0    |IIA       |
|Any T, Mets|N1         |M1    |IV        |
|T3         |N1         |M0    |IIIA      |
|T1a        |N1         |M0    |IIA       |
|T4c        |N0         |M0    |IIIB      |
|T4c        |N1         |M0    |IIIB      |
|T3         |N3         |M0    |IIIC      |
|T4a        |N1         |M0    |IIIB      |
|T4b        |N0         |M0    |IIIB      |
|Any T, Mets|N0         |M1    |IV        |
|T1a        |N0         |M0    |I         |
|T4d        |NX Adjusted|M0    |IIINOS    |
|T2         |N2         |M0    |IIIA      |
|TX Adjusted|N3         |M0    |IIIC      |
|T1b        |N3         |M0    |IIIC      |
|Tis        |NX Adjusted|MX    |

In [19]:
df.select(
    "AJCC_T"
).distinct().show()

+-----------+
|     AJCC_T|
+-----------+
|        T4d|
|         T3|
|      T1mic|
|Any T, Mets|
|        T4a|
|        T4c|
|        T1c|
|        T1b|
|         T2|
|        T1a|
|TX Adjusted|
|        Tis|
|        T4b|
|       NULL|
+-----------+



In [20]:
# ==========================================================
# 16. Standardize Category Labels
# ==========================================================

print("=" * 60)
print("STANDARDIZING CATEGORY LABELS")
print("=" * 60)

columns_to_standardize = [
    "Race",
    "Marital_Status",
    "Grade",
    "Diagnostic_Confirmation"
]

for column in columns_to_standardize:
    if column in df.columns:
        df = df.withColumn(
            column,
            F.when(
                F.col(column).isin(
                    [
                        "Blank(s)",
                        "Unknown reason",
                        "NA"
                    ]
                ),
                "Unknown"
            )
            .otherwise(
                F.col(column)
            )
        )

print("Category labels standardized")

STANDARDIZING CATEGORY LABELS
Category labels standardized


In [21]:
# ==========================================================
# 17. Detect Constant Features
# ==========================================================

print("=" * 60)
print("CONSTANT FEATURE DETECTION")
# ==========================================================

for column in df.columns:
    unique_count = (
        df
        .select(column)
        .distinct()
        .count()
    )
    if unique_count == 1:
        print(f"Constant Feature: {column}")

CONSTANT FEATURE DETECTION
Constant Feature: Sequence_Number
Constant Feature: Behavior
Constant Feature: Lymph_Vascular_Invasion


In [22]:
# ==========================================================
# 18. Remove Unnecessary Columns
# ==========================================================

print("=" * 60)
print("REMOVING UNNECESSARY COLUMNS")
print("=" * 60)

remove_columns = [
    "Patient_ID",
    "Primary_Site",
    "Behavior",
    "Lymph_Vascular_Invasion"
]

df = df.drop(
    *remove_columns
)
print("Unnecessary columns removed.")

REMOVING UNNECESSARY COLUMNS
Unnecessary columns removed.


In [23]:
# ==========================================================
# 19. Validate Clean Dataset
# ==========================================================

print("=" * 60)
print("CLEAN DATASET VALIDATION")
# ==========================================================

print(f"Rows    : {df.count():,}")

print(f"Columns : {len(df.columns)}")

df.printSchema()

CLEAN DATASET VALIDATION
Rows    : 456,087
Columns : 25
root
 |-- Age: string (nullable = true)
 |-- Sex: string (nullable = true)
 |-- Race: string (nullable = false)
 |-- Marital_Status: string (nullable = false)
 |-- Tumor_Size: integer (nullable = true)
 |-- Survival_Months: integer (nullable = true)
 |-- Vital_Status: string (nullable = true)
 |-- Grade: string (nullable = false)
 |-- PR_Status: string (nullable = true)
 |-- ER_Status: string (nullable = true)
 |-- AJCC_T: string (nullable = true)
 |-- AJCC_N: string (nullable = true)
 |-- Regional_Nodes_Examined: integer (nullable = true)
 |-- Regional_Nodes_Positive: integer (nullable = true)
 |-- Sequence_Number: string (nullable = true)
 |-- Histologic_Type: integer (nullable = true)
 |-- Laterality: string (nullable = true)
 |-- Diagnostic_Confirmation: string (nullable = false)
 |-- AJCC_M: string (nullable = true)
 |-- Surgery_Primary_Site: integer (nullable = true)
 |-- Surgery_Other_Regional: string (nullable = true)
 |--

In [24]:
# ==========================================================
# 20. Compare Before vs After Preprocessing
# ==========================================================

print("=" * 60)
print("BEFORE VS AFTER PREPROCESSING")
print("=" * 60)

before_rows = original_row_count
before_cols = original_column_count

after_rows = df.count()
after_cols = len(df.columns)

print(f"Before Rows    : {before_rows:,}")
print(f"After Rows     : {after_rows:,}")
print()

print(f"Before Columns : {before_cols}")
print(f"After Columns  : {after_cols}")
print()

print("Rows Removed   :", before_rows - after_rows)
print("Columns Removed:", before_cols - after_cols)

BEFORE VS AFTER PREPROCESSING
Before Rows    : 457,351
After Rows     : 456,087

Before Columns : 29
After Columns  : 25

Rows Removed   : 1264
Columns Removed: 4


In [ ]:
# ==========================================================
# 21. Data Cleaning Report
# ==========================================================

print("=" * 60)
print("DATA CLEANING REPORT")
print("=" * 60)


print("""
Completed preprocessing steps:
✓ Rename SEER columns
✓ Convert SEER special codes to NULL
✓ Handle missing categorical values
✓ Remove invalid records
✓ Check numerical outliers
✓ Convert data types
✓ Standardize category labels
✓ Detect constant features
✓ Remove unnecessary columns
""")

DATA CLEANING REPORT

Completed preprocessing steps:

✓ Rename SEER columns

✓ Convert SEER special codes to NULL

✓ Handle missing categorical values

✓ Remove invalid records

✓ Check numerical outliers

✓ Convert data types

✓ Standardize category labels

✓ Detect constant features

✓ Remove unnecessary columns


Dataset status:

The cleaned SEER breast cancer dataset
is ready for Feature Engineering.



In [26]:
# ==========================================================
# 22. Export Clean Dataset
# ==========================================================

output_file = "../data/processed/seer_breast_cancer_clean.csv"

df.toPandas().to_csv(
    output_file,
    index=False
)

print("=" * 60)
print("DATASET EXPORTED")
print("=" * 60)
print(output_file)

DATASET EXPORTED
../data/processed/seer_breast_cancer_clean.csv


In [30]:
# ==========================================================
# 23. Preprocessing Summary
# ==========================================================

print("=" * 60)
print("PREPROCESSING SUMMARY")
print("=" * 60)

print("\nDataset Name : SEER Breast Cancer")

print("\nOriginal Dataset")
print("-" * 16)
print(f"Rows    : {before_rows:,}")
print(f"Columns : {before_cols}")

print("\nClean Dataset")
print("-" * 16)
print(f"Rows    : {df.count():,}")
print(f"Columns : {len(df.columns)}")

print("\nChanges")
print("-" * 16)
print(f"Rows Removed    : {before_rows - df.count():,}")
print(f"Columns Removed : {before_cols - len(df.columns)}")

print("\nStatus")
print("-" * 16)
print("✓ Preprocessing completed successfully.")

print("\nOutput")
print("-" * 16)
print("The cleaned dataset is ready for")
print("Notebook 03 — Feature Engineering.")

PREPROCESSING SUMMARY

Dataset Name : SEER Breast Cancer

Original Dataset
----------------
Rows    : 457,351
Columns : 29

Clean Dataset
----------------
Rows    : 456,087
Columns : 25

Changes
----------------
Rows Removed    : 1,264
Columns Removed : 4

Status
----------------
✓ Preprocessing completed successfully.

Output
----------------
The cleaned dataset is ready for
Notebook 03 — Feature Engineering.
